## H&M Fashion Data: Data Overview
# Explanation:
This notebook contains the first examination of the H&M Personalized Fashion Recommendations dataset. The aim is to explore the three main tables (articles, customers and transactions) and examine their directions, column types, missing values and number of unique values.

# About the dataset:
The data is taken from H&M competition on Kaggle and it contains sales data between September 2018 to September 2020. The version used in this project is a copy which converted to the more memory efficient Parquet format.


## ARTICLES

In [45]:
import pandas as pd
import pyarrow.parquet as pq
import os

pd.set_option('display.max_columns', None)

In [3]:
articles = pd.read_parquet("../data/articles.parquet")
customers = pd.read_parquet("../data/customers.parquet")
transactions = pd.read_parquet("../data/transactions_train.parquet")

In [4]:
articles.shape

(105542, 25)

In [5]:
articles.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,4,Dark,5,Black,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [6]:
articles.dtypes

article_id                      int64
product_code                    int64
prod_name                         str
product_type_no                 int64
product_type_name                 str
product_group_name                str
graphical_appearance_no         int64
graphical_appearance_name         str
colour_group_code               int64
colour_group_name                 str
perceived_colour_value_id       int64
perceived_colour_value_name       str
perceived_colour_master_id      int64
perceived_colour_master_name      str
department_no                   int64
department_name                   str
index_code                        str
index_name                        str
index_group_no                  int64
index_group_name                  str
section_no                      int64
section_name                      str
garment_group_no                int64
garment_group_name                str
detail_desc                       str
dtype: object

In [7]:
articles.isnull().sum()

article_id                        0
product_code                      0
prod_name                         0
product_type_no                   0
product_type_name                 0
product_group_name                0
graphical_appearance_no           0
graphical_appearance_name         0
colour_group_code                 0
colour_group_name                 0
perceived_colour_value_id         0
perceived_colour_value_name       0
perceived_colour_master_id        0
perceived_colour_master_name      0
department_no                     0
department_name                   0
index_code                        0
index_name                        0
index_group_no                    0
index_group_name                  0
section_no                        0
section_name                      0
garment_group_no                  0
garment_group_name                0
detail_desc                     416
dtype: int64

In [8]:
articles.nunique()

article_id                      105542
product_code                     47224
prod_name                        45875
product_type_no                    132
product_type_name                  131
product_group_name                  19
graphical_appearance_no             30
graphical_appearance_name           30
colour_group_code                   50
colour_group_name                   50
perceived_colour_value_id            8
perceived_colour_value_name          8
perceived_colour_master_id          20
perceived_colour_master_name        20
department_no                      299
department_name                    250
index_code                          10
index_name                          10
index_group_no                       5
index_group_name                     5
section_no                          57
section_name                        56
garment_group_no                    21
garment_group_name                  21
detail_desc                      43404
dtype: int64

In [9]:
type_counts = articles.groupby("product_type_name")["product_type_no"].nunique()

mask = type_counts > 1

type_counts[mask]


product_type_name
Umbrella    2
Name: product_type_no, dtype: int64

In [10]:
product_type_names = articles["product_type_name"]

umbrella_mask = product_type_names == "Umbrella"

umbrellas = articles[umbrella_mask]

umbrellas.shape

umbrellas[["product_group_name", "product_type_no"]].value_counts()

product_group_name  product_type_no
Accessories         83                 26
Items               532                 3
Name: count, dtype: int64

In [11]:
def find_inconsistent_names(df, name_col, code_col):
    """
    Find the names that have multiple codes.

    Parameters
    ----------
    df : DataFrame
        The table to inspect.
    name_col : str
        Name of the column containing the names.
    code_col : str
        Name of the column containing the codes.

    Returns
    -------
    Series
        Names that have multiple codes and how many different codes they have.
"""
    code_counts = df.groupby(name_col)[code_col].nunique()
    return code_counts[code_counts > 1]


print(find_inconsistent_names(articles, "product_type_name", "product_type_no"))
print(find_inconsistent_names(articles, "section_name", "section_no"))
print(find_inconsistent_names(articles, "department_name", "department_no"))

help(find_inconsistent_names)

product_type_name
Umbrella    2
Name: product_type_no, dtype: int64
section_name
Ladies Other    2
Name: section_no, dtype: int64
department_name
Accessories                  4
Bags                         2
Belts                        2
Blazer                       2
Blouse                       3
Dress                        3
EQ & Special Collections     2
Jersey                       5
Jersey Basic                 2
Jersey Fancy                 3
Jersey License               2
Knitwear                    11
Loungewear                   2
OL Extended Sizes            2
Outwear                      2
Shirt                        4
Shorts                       5
Skirt                        2
Small Accessories            2
Swimwear                     2
Trouser                      7
Woven bottoms                2
Name: department_no, dtype: int64
Help on function find_inconsistent_names in module __main__:

find_inconsistent_names(df, name_col, code_col)
    Find the names that have

## Key Findings: Articles

1. The table contains 105,542 product variants (articles) and 25 columns. Each `article_id` is unique.
2. There are 47,224 distinct products (`product_code`). On average, each product has about 2.2 variants.
3. The only missing values are in the `detail_desc` column (416 rows, about 0.4%).
4. `article_id` is stored as `int64`, so the leading zeros are lost. It should be converted to a string in the preparation step.
5. Some names have more than one code. In particular, department names repeat across different sections (e.g., "Knitwear" has 11 different codes). Codes should be used instead of names when grouping.

## CUSTOMERS

In [12]:
customers.shape

(1371980, 7)

In [13]:
customers.head()

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,NaN,NaN,ACTIVE,NONE,54.0,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52.0,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...


In [14]:
customers.dtypes

customer_id                   str
FN                        float64
Active                    float64
club_member_status            str
fashion_news_frequency        str
age                       float64
postal_code                   str
dtype: object

In [15]:
customers.isnull().sum()
missing_percentages = (customers.isnull().sum() / len(customers) * 100).round(2)
print(missing_percentages)

customer_id                0.00
FN                        65.24
Active                    66.15
club_member_status         0.44
fashion_news_frequency     1.17
age                        1.16
postal_code                0.00
dtype: float64


In [16]:
customers["FN"]

0          NaN
1          NaN
2          NaN
3          NaN
4          1.0
          ... 
1371975    NaN
1371976    NaN
1371977    1.0
1371978    1.0
1371979    NaN
Name: FN, Length: 1371980, dtype: float64

In [17]:
customers["Active"]

0          NaN
1          NaN
2          NaN
3          NaN
4          1.0
          ... 
1371975    NaN
1371976    NaN
1371977    1.0
1371978    1.0
1371979    NaN
Name: Active, Length: 1371980, dtype: float64

In [18]:
print(customers.nunique())
print(len(customers))


customer_id               1371980
FN                              1
Active                          1
club_member_status              3
fashion_news_frequency          4
age                            84
postal_code                352899
dtype: int64
1371980


In [19]:
customers["club_member_status"].value_counts(dropna=False)

club_member_status
ACTIVE        1272491
PRE-CREATE      92960
NaN              6062
LEFT CLUB         467
Name: count, dtype: int64

In [20]:
customers["fashion_news_frequency"].value_counts(dropna=False)

fashion_news_frequency
NONE         877711
Regularly    477416
NaN           16009
Monthly         842
None              2
Name: count, dtype: int64

In [21]:
customers["age"].describe()

count    1.356119e+06
mean     3.638696e+01
std      1.431363e+01
min      1.600000e+01
25%      2.400000e+01
50%      3.200000e+01
75%      4.900000e+01
max      9.900000e+01
Name: age, dtype: float64

In [22]:
customers["postal_code"].value_counts(dropna=False)

postal_code
2c29ae653a9282cce4151bd87643c907644e09541abc28ae87dea0d1f6603b1c    120303
cc4ed85e30f4977dae47662ddc468cd2eec11472de6fac5ec985080fd92243c8       261
714976379549eb90aae4a71bca6c7402cc646ae7c40f6c1cb91d4b5a18623fc1       159
7c1fa3b0ec1d37ce2c3f34f63bd792f3b4494f324b6be5d1e4ba6a75456b96a7       157
5b7eb31eabebd3277de632b82267286d847fd5d44287ee150bb4206b48439145       156
                                                                     ...  
8a9cc8fcd8127297902050ce7b1c1f148c498ccef80e1cafd75a4ca02fc63899         1
bdcce24e89a9f810e2b10efa54e36013c3795e168ddd18b74901b716d48c48f5         1
c9d6890a6a194925f2c7712bde35b4ea973971cbe27c993ee907fe6d97a3ab0c         1
c9e32043e0338a51dfd94705cfd8f75eb98c2ec34730497c9726b21e59ee4572         1
0a1a03306fb2f62164c2a439b38c0caa64b40deaae868799ccb93ac69444b79d         1
Name: count, Length: 352899, dtype: int64

In [23]:
pd.crosstab(customers["FN"], customers["fashion_news_frequency"], dropna=False, normalize="index")

fashion_news_frequency,Monthly,NONE,None,Regularly,NaN
FN,,,,,
1.0,0.001738,0.001654,0.000004,0.996603,0.000000
NaN,0.000015,0.979746,0.000000,0.002353,0.017886


In [24]:
pd.crosstab(customers["FN"], customers["Active"], dropna=False)

Active,1.0,NaN
FN,,
1.0,464404,12526
NaN,0,895050


## Key Findings: Customers

1. The table contains 1,371,980 customers and 7 columns. Each `customer_id` is unique. IDs and postal codes are hashed for privacy reasons.
2. The `FN` and `Active` columns have about 65% missing values, but they only contain the value 1.0. The missing values most likely mean "no" and should be filled with 0. Also, every active customer is also an FN subscriber, so the two columns carry almost the same information.
3. In the `fashion_news_frequency` column, "NONE" and "None" appear as two different spellings of the same category. All missing values in this column belong to customers who are not subscribed, so they can be filled with "NONE".
4. The most frequent postal code appears for 120,303 customers (about 8.8%). This is most likely a default value and should be treated as a hidden missing value.
5. Customer ages range from 16 to 99. The mean (36.4) is greater than the median (32), which shows that the distribution is right-skewed: most customers are young.

## 3.Transactions

In [25]:
pq.read_metadata("../data/transactions_train.parquet")

  created_by: parquet-cpp-arrow version 6.0.1
  num_columns: 5
  num_rows: 31788324
  num_row_groups: 1
  format_version: 1.0
  serialized_size: 3563

In [26]:
pq.read_schema("../data/transactions_train.parquet")

t_dat: string
customer_id: string
article_id: int64
price: double
sales_channel_id: int64
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 854

In [35]:
total_memory_gb = transactions.memory_usage(deep=True).sum()/ 1024 /1024 /1024
round(total_memory_gb, 2)

np.float64(3.37)

In [42]:
memory_by_column_gb = transactions.memory_usage(deep=True)/1024/1024/1024

memory_by_column_gb

Index               1.229346e-07
t_dat               5.328933e-01
customer_id         2.131573e+00
article_id          2.368415e-01
price               2.368415e-01
sales_channel_id    2.368415e-01
dtype: float64

In [43]:
memory_by_column_gb['customer_id'] / total_memory_gb * 100

np.float64(63.15789243630587)

In [46]:
parquet_file_gb = os.path.getsize("../data/transactions_train.parquet") / 1024 / 1024 / 1024

total_memory_gb / parquet_file_gb

np.float64(4.657254690005)

In [50]:
transactions.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


In [51]:
transactions.dtypes

t_dat                   str
customer_id             str
article_id            int64
price               float64
sales_channel_id      int64
dtype: object

In [52]:
transactions.isnull().sum()

t_dat               0
customer_id         0
article_id          0
price               0
sales_channel_id    0
dtype: int64

In [56]:
transactions["t_dat"].min(), transactions["t_dat"].max()

('2018-09-20', '2020-09-22')

In [ ]:
customers["customer_id"].nunique(), transactions["article_id"].nunique()

KeyError: 'article_id'